# Fault Tolerance

Kafi Streams ensures fault tolerance with checkpointing.

We explain how it works in [Checkpointing](#checkpointing).

Finally, we show checkpointing in action by a practical [example](#example).

Notice that fault tolerance is only supported by *Streams*, not the *TopologyNode* class as only the former includes support for persistence (through Kafi).


## Overview

[Preparation](#prep)

* [Checkpointing](#checkpointing)
  * [Enabling checkpointing](#enabling)
  * [Checkpointing in detail](#detail)
* [Example](#example)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [1]:
import sys
sys.path.insert(1, "../..")

from kafi.streams.streams import Streams

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"


---
<a id="checkpointing"></a>
## Checkpointing

How does checkpointing work in Kafi Streams?

This global state can be stored in any *storage* supported by Kafi, i.e., currently, Kafka itself, or, via Kafi's *emulated Kafka*, to disk, S3 or Azure Blob Storage.

[Enabling checkpointing](#enabling) explains how to enable checkpointing, and [Checkpointing in detail](#detail) describes how checkpointing is implemented in detail and embedded into the consume + process + produce loop of the *Streams* class.


<a id="enabling"></a>
### Enabling checkpointing

Let's revisit the signature of the `start_streams` method of the *Streams* class to see how checkpointing can be enabled:
```python
@staticmethod
def start_streams(built_tn, checkpoint_storage=None, checkpoint_topic_str=None, checkpoint_interval_float=default_checkpoint_interval_float, **kwargs):
    """Run streams() in a background thread; returns a function to stop it.

    Args:
        built_tn: built tn to run
        checkpoint_storage: storage backend for checkpoints, or None to disable checkpointing
        checkpoint_topic_str: topic name used to store checkpoints
        checkpoint_interval_float: seconds between checkpoints
        **kwargs: passed through to streams()
    Returns:
        stop_fun: None -> None function to stop the Streams processing thread"""
```

You can enable checkpointing by setting `checkpoint_storage` and `checkpoint_topic_str` to the Kafi stroage and topic to be used for the checkpointing.

The `checkpoint_interval_float` is a floating point number specifying the checkpoint interval (in seconds).


<a id="detail"></a>
### Checkpointing in detail

*Streams* implements the typical consume + process + produce loop from stream processing. Let's look at it in more detail in the case if checkpointing is enabled:

* before the loop: read the last checkpoint if there is any and set the global state and the offsets of the consumer group for the source topics accordingly,
* in the loop:
  1. consume new data from the source topics,
  2. process the new data and return the new resulting data,
  3. produce the new resulting data to the sink topics.
  4. if there is new resulting data and the checkpoint interval is exceeded:  
     4.1 save the checkpoint = the current global state + the offsets of the last processed messages from the source topics,  
     4.2 commit these offsets to Kafka.


---
<a id="example"></a>
## Example

This section shows an example of checkpointing in Kafi Streams.


<a id="topology"></a>
### Topology

The topology has one source (orders) and aggregates these orders per `customer_id`:
* `orders` the number of orders of the customer,
* `order_ids` the `order_id`s of the customer,
* `total_price` the sum of the `price`s of the orders of the customer:


In [2]:
import sys
sys.path.insert(1, "../..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.DEBUG)

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

sink_tn = (
    Streams.source(c, source_str)

    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["order_id"]]),
                                    "total_price": agg_r["total_price"] + r["price"]},
                  {"orders": 0, "order_ids": [], "total_price": 0},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "total_price": agg_r["total_price"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

tn = Streams.build(sink_tn)



<a id="step_1"></a>
### Step 1

In step 1, we:
* start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [3]:
from kafi.helpers import get_millis

orders_int = 1000

tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, checkpoint_interval_floatrval=0.01, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



INFO:kafi.streams.streams:Starting Streams...
DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787588049618_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1787588049618_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

'orders'

DEBUG:kafi.streams.streams:Source consumer group ('group_1787588049618') offsets for topic 'orders': {}



(['orders'], 'group_1787588049618')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (62 KB compressed, 382 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 1000} for source orders.


sink: {'key': 7, 'value': {'customer_id': 7, 'orders': 97, 'order_ids': ['00039b68-9061-4c91-8bd2-6290fdf140b9', '0042c6b7-ab91-4e80-82f0-4a640d661dc8', '051ce98c-7199-442e-90cd-7533d84c976e', '06a1230c-785e-4ecf-a88c-7f349e478655', '0d97fa6f-4eb1-4ce8-9460-813044fd1607', '1220942c-874b-4abf-a064-e0a8a3a9586d', '12edbfaa-5f4c-43da-be1b-8affe37adbf1', '18311b83-6237-4a6d-b4b9-a5b3dee593a0', '185e277c-1ab8-4a42-a194-df6905659348', '1f52cce7-a2a3-4b94-a2e3-642749101680', '20675742-1c72-48b6-98e8-d4a0bbb3c87a', '25800b16-d151-472a-aeb8-59d7cd6501c5', '28e7273d-5061-4b6d-9617-58d1ea57368e', '29b3aeb3-3bfe-42d3-a4cf-4121fe813860', '2bad281f-dd10-4626-a650-9cd542a5fae9', '2d8848a0-abf3-4de3-8a14-1458281608b5', '318cec56-13a3-4585-a6c7-161ea72dc384', '32e0006d-7a80-41b6-b193-22f189cc4597', '35d3ef0c-c718-49ff-97d2-9a64b93706e9', '373cbd6e-4b60-458a-82b6-0631dad1a510', '38a9f28c-6f96-4be4-83cb-dea531a1d1ba', '3a001b44-d9a8-4662-a953-f8c5e3596265', '3b06333d-d95f-47d6-aeca-fdb6ba5e7a64', '3e1655

As you see in the output, a first checkpoint has been saved to the checkpoint topic.

<a id="step_2"></a>
### Step 2

In step 2, we stop the *Streams* thread:

In [4]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_3"></a>
### Step 3

In step 3, we:
* reset the state of the built topology node (`built_tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [ ]:
tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



INFO:kafi.streams.streams:Starting Streams...
DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787588049618_checkpoint') offsets for topic 'checkpoint': {}


{'orders': 1000}
{'orders_aggregated': 10}
{'checkpoint': 64}
(['checkpoint'], 'group_1787588049618_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

'orders'

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (62 KB compressed, 382 KB uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1787588049618') offsets for topic 'orders': {0: 1000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 1000}



(['checkpoint'], 'group_1787588049618_checkpoint')
(['orders'], 'group_1787588049618')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (122 KB compressed, 886 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 2000} for source orders.


sink: {'key': 4, 'value': {'customer_id': 4, 'orders': 199, 'order_ids': ['0233adab-25b6-4919-8144-a2a9728aeccb', '026a2b9a-2fb1-4a34-ad98-968f8dfe0221', '02e5c039-904c-4340-9370-2d40e7ff283b', '0318db59-8d9b-44a3-99b9-8c0418f321f5', '0796e44b-0984-46d1-aa30-666fca41470d', '08c152be-a9c3-4fcd-ad85-326dc8ad00f9', '09fc1636-2853-4a5f-b2d8-9b470968c52b', '0a78d578-b899-4577-889b-9b7d26b2f073', '0bc2739c-40e6-448c-a0bf-038f5afea428', '13aa6a70-1a60-4475-8259-625c7ae448d4', '14e179ca-3e9a-4d5d-8c99-4ed27cff72f9', '1927302f-08ca-4e4f-8aaf-e981dbecd248', '19f08b2b-8258-4904-92e8-b176d057f284', '1dda17aa-cb8d-4340-b7ba-a0a4d07e2dab', '1e656fee-6d4e-48c2-9337-3858b2f3c63a', '1ed75067-cf25-45c4-8df1-67152733ae2e', '20bc86a3-59e3-4757-812a-39caa8034299', '20da5036-35bb-4113-ad4a-4b27052f212b', '21391a1e-ec73-41f0-8262-1da7b82c4ab9', '2251b611-9f07-44b6-8f41-cac9f5108cc2', '23dddc20-d7c5-4c2e-ad68-75e312384003', '244ead73-0d75-420d-8a44-6e65e28ce932', '270ffa5e-f0fb-4d69-b5d6-b7e9d0cd91fa', '272e9

In the output log, you can see that the checkpoint from the previous processing ([step 1](#step_1)) is correctly loaded and thus the state recovered.

<a id="step_4"></a>
### Step 4

In step 4, we stop the *Streams* thread once again:

In [6]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_5"></a>
### Step 5

In step 5, we again:
* reset the state of the built topology node (`built_tn`),
* (re-)start the *Streams* thread using the `start_streams()` method (with checkpointing enabled),
* generate another `1000` orders, produce them to the source topic and let *Streams* thread process them:

In [ ]:
tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams(tn, checkpoint_storage=c, checkpoint_topic_str=checkpoint_str, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


INFO:kafi.streams.streams:Starting Streams...
DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1787588049618_checkpoint') offsets for topic 'checkpoint': {0: 64}


{'orders': 2000}
{'orders_aggregated': 20}
{'checkpoint': 190}
(['checkpoint'], 'group_1787588049618_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

'orders'

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (122 KB compressed, 886 KB uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1787588049618') offsets for topic 'orders': {0: 2000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 2000}



(['checkpoint'], 'group_1787588049618_checkpoint')
(['orders'], 'group_1787588049618')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (167 KB compressed, 1198 KB uncompressed).
INFO:kafi.streams.streams:Committed {0: 3000} for source orders.


sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 303, 'order_ids': ['0003579a-4c8e-42c4-bcbb-97762c4bce2a', '009d3594-68c0-432a-9240-55190956b44d', '00bba0b0-fb2b-4a00-8a2f-3b5727b2ea31', '0253a1e6-9b94-4cb6-b438-76ca80bae43e', '03cebc20-12b1-49b3-bc19-4924482b54b2', '057832e8-abd8-4c60-b7e0-eb7dc00122ce', '08907ec7-50b6-4641-a7f1-b54fd853c195', '0917849c-f4d7-4e1b-8b36-921708385a72', '09d2eff2-19e3-486d-893a-f6201fd68f15', '0c97fc9b-2d70-4f9d-a8ac-863508fabbd6', '0ce4bb39-f163-4c1c-8f02-10f31af2243e', '0df6471c-1b60-4e3f-af0c-2035b50abdc1', '10ca1a14-21b3-4903-b749-19c4c033e596', '1289753d-4368-4414-9cba-b46d3f7dee11', '12c8fbae-0b40-4ba2-9ee5-894b200f1926', '13244719-1cd6-4af7-ba4a-e5dadff31a3a', '137826b5-c309-449a-aa94-50211881af74', '1445b7e1-203a-4b7a-8b65-02fed56f5654', '15b31c78-d8e5-4f87-ad7c-8156e864a003', '1739bdc4-eb16-4e46-883e-866e8283b231', '17801f59-c54c-4617-ade3-f5de4cadf121', '184bf5ba-5927-4216-b7d7-6343e5708f1e', '19ca28be-d1e8-4444-ae3c-7a91bace5dc5', '1a3b0

<a id="step_6"></a>
### Step 6

In step 6, we stop the *Streams* thread a last time:


In [8]:
stop_fun()
Streams.threads()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

<a id="step_7"></a>
### Step 7

In the last step 7, we:
* read both the source and the sink topic,
* calculate the aggregations outside of Kafi Streams,
* and compare them to the aggregations read from the sink topic:

In [9]:
import math

source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    customer_id_int = m["value"]["customer_id"]
    order_id_str = m["value"]["order_id"]
    price_int = m["value"]["price"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_str_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_total_price_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("total_price", 0)
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_str_list + [order_id_str]),
                                                       "total_price": agg_total_price_int + price_int}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

for key_int, value_dict in source_key_int_value_dict_dict.items():
    keys_match_bool = value_dict["customer_id"] == sink_key_int_value_dict_dict[key_int]["customer_id"]
    orders_match_bool = value_dict["orders"] == sink_key_int_value_dict_dict[key_int]["orders"]
    order_ids_match_bool = value_dict["order_ids"] == sink_key_int_value_dict_dict[key_int]["order_ids"]
    price_match_bool = math.isclose(
        value_dict["total_price"], sink_key_int_value_dict_dict[key_int]["total_price"], rel_tol=1e-9)
    #
    if not (keys_match_bool and orders_match_bool and order_ids_match_bool and price_match_bool):
        print("First mismatch:")
        print("Source:", source_dict)
        print("Sink:  ", sink_dict)
        raise Exception("Test failed")
#

print("Test successful.")


(['orders'], '1787588113292')


Consuming: 0 msg [00:00, ? msg/s]


(['orders_aggregated'], '1787588118552')


Consuming: 0 msg [00:00, ? msg/s]


{7: {'customer_id': 7, 'orders': 285, 'order_ids': ['00039b68-9061-4c91-8bd2-6290fdf140b9', '0042c6b7-ab91-4e80-82f0-4a640d661dc8', '0091e917-b69a-40f8-99f0-e2908138b883', '01177367-e000-43b3-b74f-660edaf2e88d', '01d3b34d-0120-4392-882b-786065d6a1e8', '03941a6d-a844-4200-a9cf-e823b4c97939', '051ce98c-7199-442e-90cd-7533d84c976e', '05aaebc1-bc93-4fdd-a1d6-abcd453ae21a', '06a1230c-785e-4ecf-a88c-7f349e478655', '0a5d3168-5580-4c41-a6d4-1381589154a0', '0c2f834d-a9f6-47fb-9864-a1bc135cf48d', '0c52b53a-215d-4647-a0f6-da3f66a9d47d', '0d97fa6f-4eb1-4ce8-9460-813044fd1607', '0e4c37e4-e04b-4146-90c4-e2b5ef77641c', '0e8ff327-e6b2-4879-b2ec-b9cd32902a8c', '0edb1b07-cc33-4dd3-b91f-66d64000abbd', '0fa66bd5-718f-430d-9ba6-64c10a4d3c85', '11345914-3481-4c5b-9d8a-683aedd59788', '115409d9-6080-4d70-92c1-f4e38d66bdb5', '1220942c-874b-4abf-a064-e0a8a3a9586d', '12edbfaa-5f4c-43da-be1b-8affe37adbf1', '1484c907-7456-4b69-aac6-7045e5e2eb9d', '156db72a-c67f-4879-a59c-dc553c432e85', '15f85b4b-39f8-4d42-8130-26

Because in this step, we read the entire source topic which has been subsequently filled with new data after we had intentionally stopped the *Streams* thread, and the entire sink topic, and because all the source data is randomly generated, this steps show that the checkpointing in Kafi Streams works :)
